In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from dotenv import load_dotenv
import os

In [2]:
load_dotenv("../.env")
census_key = os.getenv("CENSUS_API_KEY")

In [3]:
API_KEY    = census_key
STATE_FIPS = "54"                          
YEARS      = [2018, 2019, 2020, 2021, 2022, 2023]

In [4]:
def fetch_acs(year, table, variables):
    url = (
        f"https://api.census.gov/data/{year}/acs/acs5/subject"
        f"?get=NAME,{variables}"
        f"&for=county:*"
        f"&in=state:{STATE_FIPS}"
        f"&key={API_KEY}"
    )
    response = requests.get(url)
    response.raise_for_status()
    data = response.json()
    headers = data[0]
    records = [dict(zip(headers, row)) for row in data[1:]]
    df = pd.DataFrame(records)
    df["Year"] = year
    df["FIPS_Code"] = df["state"] + df["county"]
    df["County"] = df["NAME"].str.replace(", West Virginia", "", regex=False)
    return df

In [5]:
def fetchacs5(year, variables):
    url = f"https://api.census.gov/data/{year}/acs/acs5"
    
    params = {
        "get": f"NAME,{variables}",
        "for": "county:*",
        "in": f"state:{STATE_FIPS}",
        "key": API_KEY
    }

    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()

    df = pd.DataFrame(data[1:], columns=data[0])
    df["Year"] = year
    df["FIPS_Code"] = df["state"] + df["county"]
    df["County"] = df["NAME"].str.replace(", West Virginia", "", regex=False)

    # Convert requested variables to numeric
    for var in variables.split(","):
        df[var] = pd.to_numeric(df[var], errors="coerce")
    
    return df

In [6]:
poverty_records = []

for year in YEARS:
    df_pov = fetch_acs(year, "S1701", "S1701_C03_001E,S1701_C03_002E")
    df_pov["Poverty_Percent_All_Ages"] = pd.to_numeric(df_pov["S1701_C03_001E"], errors="coerce")
    df_pov["Poverty_Percent_Age_0_17"] = pd.to_numeric(df_pov["S1701_C03_002E"], errors="coerce")
    poverty_records.append(df_pov[["Year", "FIPS_Code", "County",
                                    "Poverty_Percent_All_Ages", "Poverty_Percent_Age_0_17"]])

df_poverty = pd.concat(poverty_records, ignore_index=True)


# ── STEP 2: Pull S1903 — Median Household Income ─────────────────────────────
# S1903_C02_001E = Median household income
income_records = []

for year in YEARS:
    df_inc = fetchacs5(year, "B19013_001E")  # use ACS5 detailed tables
    df_inc["Median_Household_Income"] = pd.to_numeric(df_inc["B19013_001E"], errors="coerce")
    df_inc["Year"] = year
    df_inc["FIPS_Code"] = df_inc["state"] + df_inc["county"]
    income_records.append(df_inc[["Year", "FIPS_Code", "Median_Household_Income"]])

df_income = pd.concat(income_records, ignore_index=True)


# ── STEP 3: Merge and finalize ────────────────────────────────────────────────
df = pd.merge(df_poverty, df_income, on=["Year", "FIPS_Code"], how="left")
df = df[["Year", "FIPS_Code", "County", "Poverty_Percent_All_Ages",
         "Poverty_Percent_Age_0_17", "Median_Household_Income"]].copy()
df = df.sort_values(["Year", "FIPS_Code"]).reset_index(drop=True)

df

,Year,FIPS_Code,County,Poverty_Percent_All_Ages,Poverty_Percent_Age_0_17,Median_Household_Income
0,2018,54001,Barbour County,23.0,34.7,39580
1,2018,54003,Berkeley County,12.7,18.9,60615
2,2018,54005,Boone County,24.3,35.6,38642
3,2018,54007,Braxton County,21.8,35.8,42213
4,2018,54009,Brooke County,13.6,20.5,49772
...,...,...,...,...,...,...
325,2023,54101,Webster County,22.1,32.4,42061
326,2023,54103,Wetzel County,16.6,16.7,53341
327,2023,54105,Wirt County,18.3,30.4,54688
328,2023,54107,Wood County,14.8,19.0,56193


In [7]:
df.to_csv("poverty_and_income.csv", index=False)